# Étteremértékelések elemzése és ajánlórendszer – projekt notebook

**Tárgy:** Mesterséges intelligencia alkalmazási területei  
**Benchmark adathalmaz:** SemEval-2016 Task 5 – English Restaurants, Subtask 1

## Ennek a notebooknak a célja

Ebben az első notebookban még **nem tanítunk Transformer-modellt**. Először az adatokat készítjük elő úgy, hogy később stabilan lehessen rájuk modelleket építeni.

A mostani mérföldkő négy részből áll:

1. a SemEval XML-fájlok betöltése és feldolgozása;
2. az adathalmaz alapvető ellenőrzése;
3. két külön modellezési tábla létrehozása:
   - `aspect_df`: mondatszintű, többcímkés aspektusfelismeréshez;
   - `sentiment_df`: aspektusszintű sentiment-osztályozáshoz;
4. a hivatalos gold test adathalmaz érintetlenül hagyása a végső értékeléshez.

### Miért készítünk két külön táblát?

Egy éttermi értékelés egyetlen mondatában több szempont is szerepelhet, például:

> *“The food was excellent, but the service was slow.”*

Ebben egyszerre jelenik meg a **FOOD#QUALITY** és a **SERVICE#GENERAL** aspektus. Emiatt az aspektusfelismerés egy **multi-label classification** feladat.

A sentiment esetében viszont már egy konkrét aspektusról szeretnénk eldönteni, hogy pozitív, semleges vagy negatív vélemény tartozik-e hozzá. Ezért ezt külön modellezési problémaként kezeljük.


## 1. Környezet beállítása

Ebben a szakaszban csak standard Python-csomagokra van szükségünk:

- `xml.etree.ElementTree`: az XML-fájlok feldolgozásához;
- `pandas`: a táblázatos adatok kezeléséhez;
- `numpy`: egyszerű numerikus ellenőrzésekhez.

A Hugging Face / Transformer csomagokat csak a következő mérföldkőnél adjuk hozzá, amikor az adat-előkészítés már stabilan működik.


In [ ]:
import xml.etree.ElementTree as ET
from pathlib import Path
from collections import Counter

import pandas as pd
import numpy as np

pd.set_option("display.max_colwidth", 160)
pd.set_option("display.max_columns", 30)

RANDOM_STATE = 42


## 2. XML-fájlok feltöltése Google Colabba

A következő cella futtatásakor töltsd fel ezt a két fájlt:

- `ABSA16_Restaurants_Train_SB1_v2.xml`
- `EN_REST_SB1_TEST.xml.gold`

Az első a hivatalos **training** adathalmaz, a második a hivatalos, gold címkékkel ellátott **test** adathalmaz.

Fontos módszertani szabály: a test fájlt később **nem használjuk modellválasztásra vagy hangolásra**. A train fájlból fogunk külön train és validation részt készíteni, a gold test pedig csak a végső értékeléshez kell.


In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
except ImportError:
    print("Nem Google Colab környezetben fut a notebook – a kézi feltöltést kihagyjuk.")


In [ ]:
TRAIN_PATH = Path("ABSA16_Restaurants_Train_SB1_v2.xml")
TEST_PATH = Path("EN_REST_SB1_TEST.xml.gold")

assert TRAIN_PATH.exists(), f"Hiányzó fájl: {TRAIN_PATH}"
assert TEST_PATH.exists(), f"Hiányzó fájl: {TEST_PATH}"

print("Mindkét XML-fájl elérhető.")


## 3. XML-fájlok feldolgozása

A SemEval adatok hierarchikusan vannak tárolva:

**Review → Sentence → Opinion**

Ezért három külön táblát építünk:

- `reviews`: egy sor = egy teljes review;
- `sentences`: egy sor = egy mondat;
- `opinions`: egy sor = egy annotált vélemény.

Egy mondathoz több `Opinion` is tartozhat. Egy opinion fő mezői:

- `target`: mire vonatkozik a vélemény, pl. `food`, `service`, `wine list`;
- `category`: az aspektuskategória, pl. `FOOD#QUALITY`;
- `polarity`: `positive`, `neutral` vagy `negative`;
- `from`, `to`: a target karakterpozíciója a mondatban.

Az `OutOfScope="TRUE"` jelölésű mondatokat megtartjuk az ellenőrzéshez, de a modellezésből később kizárjuk.


In [ ]:
def parse_semeval_xml(path):
    root = ET.parse(path).getroot()

    reviews = []
    sentences = []
    opinions = []

    for review in root.findall("Review"):
        review_id = review.attrib["rid"]
        reviews.append({"review_id": review_id})

        sentences_node = review.find("sentences")
        if sentences_node is None:
            continue

        for sentence in sentences_node.findall("sentence"):
            sentence_id = sentence.attrib["id"]
            text = sentence.findtext("text", default="")
            out_of_scope = sentence.attrib.get("OutOfScope", "FALSE").upper() == "TRUE"

            sentences.append({
                "review_id": review_id,
                "sentence_id": sentence_id,
                "text": text,
                "out_of_scope": out_of_scope,
            })

            opinions_node = sentence.find("Opinions")
            if opinions_node is None:
                continue

            for opinion in opinions_node.findall("Opinion"):
                opinions.append({
                    "review_id": review_id,
                    "sentence_id": sentence_id,
                    "text": text,
                    "target": opinion.attrib.get("target"),
                    "category": opinion.attrib.get("category"),
                    "polarity": opinion.attrib.get("polarity"),
                    "from": int(opinion.attrib.get("from", 0)),
                    "to": int(opinion.attrib.get("to", 0)),
                    "out_of_scope": out_of_scope,
                })

    return (
        pd.DataFrame(reviews),
        pd.DataFrame(sentences),
        pd.DataFrame(opinions),
    )


In [ ]:
train_reviews, train_sentences, train_opinions = parse_semeval_xml(TRAIN_PATH)
test_reviews, test_sentences, test_opinions = parse_semeval_xml(TEST_PATH)

print("TRAIN ADATHALMAZ")
print("Review-k száma:", train_reviews["review_id"].nunique())
print("Mondatok száma:", len(train_sentences))
print("Out-of-scope mondatok:", train_sentences["out_of_scope"].sum())
print("Opinion annotációk:", len(train_opinions))

print("\nTEST ADATHALMAZ")
print("Review-k száma:", test_reviews["review_id"].nunique())
print("Mondatok száma:", len(test_sentences))
print("Out-of-scope mondatok:", test_sentences["out_of_scope"].sum())
print("Opinion annotációk:", len(test_opinions))


### Elvárt darabszámok

Ha az XML-feldolgozás helyesen működik, nagyjából a következő értékeket kell kapnunk:

| Adathalmaz | Review-k | Mondatok | OutOfScope mondatok | Opinion annotációk |
|---|---:|---:|---:|---:|
| Train | 350 | 2 000 | 63 | 2 507 |
| Gold test | 90 | 676 | 34 | 859 |

Ha ezek az értékek jelennek meg, az jó ellenőrzés arra, hogy a két fájlt megfelelően olvastuk be.


## 4. Az adatok első áttekintése

Mielőtt modellt építünk, meg kell érteni az adathalmaz szerkezetét és eloszlásait.

Ebben a részben megnézzük:

- milyen aspektuskategóriák fordulnak elő;
- milyen gyakoriak az egyes aspektusok;
- hogyan oszlanak meg a pozitív, semleges és negatív sentimentek.

Ez azért fontos, mert az erősen ritka kategóriák és az osztály-aránytalanság később hatással lesz a modell teljesítményére és az értékelési metrikák megválasztására.


In [ ]:
aspect_counts = (
    train_opinions["category"]
    .value_counts()
    .rename_axis("aspect")
    .reset_index(name="count")
)

sentiment_counts = (
    train_opinions["polarity"]
    .value_counts()
    .rename_axis("sentiment")
    .reset_index(name="count")
)

display(aspect_counts)
display(sentiment_counts)


In [ ]:
print("Aspektuskategóriák száma:", train_opinions["category"].nunique())
print(sorted(train_opinions["category"].unique()))


### Fontos megfigyelés: kiegyensúlyozatlan sentiment-osztályok

A sentiment címkék eloszlása nem egyenletes. A **neutral** osztály jóval ritkább, mint a pozitív vagy negatív példák.

Ezért később nem elég kizárólag accuracy-t használni. A modell értékelésénél figyelni fogjuk:

- precision;
- recall;
- F1-score;
- **macro-F1**, amely minden osztályt azonos súllyal vesz figyelembe.

A macro-F1 különösen fontos akkor, ha egy ritka osztály – például a neutral – teljesítményét sem szeretnénk „elrejteni” a nagyobb osztályok mögött.


## 5. `aspect_df` létrehozása – aspektusfelismerési adathalmaz

Ez lesz az **aspect detection** modellünk tanítóadata.

A tábla logikája:

- egy sor = egy nem OutOfScope mondat;
- minden aspektuskategóriához tartozik egy külön 0/1 változó;
- egy mondat egyszerre több aspektust is tartalmazhat;
- azok a mondatok is bent maradnak, amelyekhez nincs annotált aspektus.

Példa:

> *“Great food, but slow service.”*

A megfelelő címkék lehetnek:

- `FOOD#QUALITY = 1`
- `SERVICE#GENERAL = 1`
- minden más aspektus = 0

Mivel egyszerre több címke is lehet 1, ez **multi-label classification**, nem hagyományos multiclass classification.


In [ ]:
ASPECTS = sorted(train_opinions["category"].unique())

def build_aspect_df(sentences_df, opinions_df, aspects):
    df = sentences_df.loc[~sentences_df["out_of_scope"]].copy()

    sentence_aspects = (
        opinions_df.loc[~opinions_df["out_of_scope"]]
        .groupby("sentence_id")["category"]
        .apply(set)
        .to_dict()
    )

    for aspect in aspects:
        df[aspect] = df["sentence_id"].map(
            lambda sid: int(aspect in sentence_aspects.get(sid, set()))
        )

    return df.reset_index(drop=True)

train_aspect_df = build_aspect_df(train_sentences, train_opinions, ASPECTS)
test_aspect_df = build_aspect_df(test_sentences, test_opinions, ASPECTS)

print("Train aspect rows:", len(train_aspect_df))
print("Test aspect rows:", len(test_aspect_df))

display(train_aspect_df.head())


In [ ]:
train_aspect_df["n_aspects"] = train_aspect_df[ASPECTS].sum(axis=1)

print("Annotált aspektus nélküli mondatok száma:",
      (train_aspect_df["n_aspects"] == 0).sum())

print("\nAspektusok száma mondatonként:")
display(train_aspect_df["n_aspects"].value_counts().sort_index())


### Elvárt eredmények az aspektustáblában

A train fájlból várhatóan:

- **1 937** modellezhető, nem OutOfScope mondat marad;
- ebből **229** olyan mondat, amelyhez nincs annotált aspektus.

Ezeket a „nulla címkés” mondatokat nem dobjuk el. Fontos negatív példák: a modellnek azt is meg kell tanulnia, hogy nem minden mondat tartalmaz releváns éttermi aspektust.


## 6. `sentiment_df` létrehozása – aspektusszintű sentiment-osztályozás

A második modellezési táblában már nem mondatonként, hanem **opinion annotációnként** dolgozunk.

A későbbi modell bemenete három információt fog tartalmazni:

- a teljes mondatot;
- a `target` értékét;
- az aspektuskategóriát.

A célváltozó:

- `positive`
- `neutral`
- `negative`

Például:

> Sentence: *“The food was excellent, but the service was slow.”*  
> Target: `service`  
> Aspect: `SERVICE#GENERAL`

→ elvárt sentiment: `negative`

### Miért használjuk a targetet is?

Ugyanabban a mondatban ugyanazon aspektuskategórián belül is előfordulhat több külön target eltérő sentimenttel. Emiatt a `sentence + aspect` önmagában nem mindig azonosítja egyértelműen a véleményt.

### Ellentmondásos címkék kezelése

Nagyon kevés olyan eset van, ahol ugyanahhoz a `(sentence, target, aspect)` kombinációhoz több különböző polarity tartozik. Ezeket az első, stabil modellverzióból kizárjuk, és ezt a dokumentációban külön adat-előkészítési döntésként rögzítjük.


In [ ]:
def build_sentiment_df(opinions_df):
    df = opinions_df.loc[~opinions_df["out_of_scope"]].copy()

    key_cols = ["sentence_id", "target", "category"]

    # Először eltávolítjuk a teljesen azonos, duplikált annotációkat.
    df = df.drop_duplicates(
        subset=key_cols + ["polarity"]
    ).copy()

    # Megkeressük azokat az eseteket, ahol ugyanahhoz a
    # (mondat, target, aspektus) kulcshoz több különböző sentiment tartozik.
    polarity_nunique = (
        df.groupby(key_cols)["polarity"]
        .nunique()
        .rename("n_polarities")
        .reset_index()
    )

    ambiguous_keys = polarity_nunique.loc[
        polarity_nunique["n_polarities"] > 1,
        key_cols
    ]

    # Az első stabil modellverzióból kizárjuk az ellentmondásos eseteket.
    if len(ambiguous_keys) > 0:
        ambiguous_marker = ambiguous_keys.assign(ambiguous=True)
        df = df.merge(ambiguous_marker, on=key_cols, how="left")
        df = df.loc[df["ambiguous"].isna()].drop(columns="ambiguous")

    return df.reset_index(drop=True), ambiguous_keys

train_sentiment_df, train_ambiguous = build_sentiment_df(train_opinions)
test_sentiment_df, test_ambiguous = build_sentiment_df(test_opinions)

print("Train sentiment sorok:", len(train_sentiment_df))
print("Eltávolított bizonytalan train kulcsok:", len(train_ambiguous))
print("Test sentiment sorok:", len(test_sentiment_df))
print("Eltávolított bizonytalan test kulcsok:", len(test_ambiguous))

display(train_sentiment_df[
    ["sentence_id", "text", "target", "category", "polarity"]
].head(10))


### Elvárt eredmények a sentiment táblában

Az ellentmondásos `(sentence, target, aspect)` esetek eltávolítása után várhatóan:

- train: **2 491** használható sentiment-példa;
- test: **853** használható sentiment-példa;
- eltávolított bizonytalan kulcsok: **7 train**, **3 test**.

A `target="NULL"` értékeket **nem töröljük**. Ezek implicit véleményeket jelentenek: a szöveg alapján az aspektus és a sentiment egyértelmű, de nincs konkrét szövegrész targetként megjelölve.


In [ ]:
print("Train sentiment eloszlás:")
display(
    train_sentiment_df["polarity"]
    .value_counts()
    .to_frame("darabszám")
)

print("\nPéldák implicit véleményekre (target = NULL):")
display(
    train_sentiment_df.loc[
        train_sentiment_df["target"] == "NULL",
        ["text", "target", "category", "polarity"]
    ].head(10)
)


## 7. Ellenőrzések a modellezés előtt

A következő cella néhány automatikus sanity checket futtat.

Ellenőrizzük, hogy:

- az `OutOfScope` mondatok valóban kikerültek a modellezési táblákból;
- az aspektuscímkék tényleg csak 0 vagy 1 értékeket tartalmaznak;
- a sentiment címkék csak `positive`, `neutral`, `negative` értékek lehetnek.

Ezek egyszerű ellenőrzések, de segítenek elkerülni, hogy hibás adatokkal kezdjünk modellt tanítani.


In [ ]:
assert not train_aspect_df["out_of_scope"].any()
assert not test_aspect_df["out_of_scope"].any()

assert set(np.unique(train_aspect_df[ASPECTS].values)).issubset({0, 1})

assert set(train_sentiment_df["polarity"].unique()).issubset(
    {"positive", "neutral", "negative"}
)

print("Az ellenőrzések sikeresen lefutottak.")


## Következő mérföldkő

Ha ez a notebook hiba nélkül lefut, az **adat-előkészítés első verziója elkészült**.

A következő notebookban / szakaszban:

1. a hivatalos training adatokból **review-szintű train / validation splitet** készítünk;
2. tokenizáljuk a mondatokat;
3. betöltünk egy előtanított Transformer encodert;
4. elkészítjük az első **multi-label aspect detection** modellt;
5. a validation adaton precision, recall és F1 metrikákkal értékeljük;
6. a hivatalos gold test adathalmazhoz csak akkor nyúlunk, amikor a modell és a beállítások már véglegesek.

### Miért review-szintű split?

Ha ugyanannak a review-nak egyes mondatai trainbe, más mondatai validationbe kerülnének, a két rész túl hasonló információt tartalmazhatna. Ez adat-szivárgást (data leakage) okozhatna, és túl optimista validációs eredményt adna.

Ezért egy review összes mondata mindig ugyanabba a részhalmazba fog kerülni.
